# Phase 5 (LSI) - Ensemble of the finished runs
No training and no GPU. It reads the out-of-fold (OOF) and test probabilities saved by the finished notebooks, combines them, and tunes the weights and thresholds on the OOF documents. Every choice is scored on documents it was not tuned on (held-out check), so the number printed for the winner is an honest estimate.

**Attach (Kaggle "Add Input"):**
1. The LSI data-prep output (`tokenized_lsi.pkl`, `lsi_utils.py`)
2. The output of each finished run (each output folder holds `oof_probs_fold0.npy`, `oof_labels_fold0.npy`, `oof_doc_ids_fold0.json`, `test_probs_fold0.npy`): `lsi_m1_lora`, `lsi_m1_lorma`, `lsi_m4_lora`, `lsi_m4_lorma`, and the distilled student `lsi_kd_m1` if you have it. Any subset of at least 2 works.

All runs must come from the same data bundle and the same folds, so their OOF documents line up. The notebook checks this by document id.

**Output:** `/kaggle/working/lsi_ensemble/` with `submission.zip` (holds `prediction.json`), `prediction.json`, `submission_lsi_ensemble.json`, `ensemble_report.json`, `per_label_thresholds.json`.

## 1. Configuration

In [ ]:
import os
import sys
import json
import time
import shutil
import pickle
import zipfile

import numpy as np

# Env vars are only for local testing. On Kaggle leave them alone.
DATA_DIR = os.environ.get("LSI_DATA_DIR", None)
INPUT_ROOT = os.environ.get("LSI_INPUT_ROOT", "/kaggle/input")
OUT_DIR = os.environ.get("LSI_OUT_DIR", "/kaggle/working/lsi_ensemble")

# Folder names of the finished runs (the name of the output folder inside each attached notebook output).
MODEL_FOLDERS = ["lsi_m1_lora", "lsi_m1_lorma", "lsi_m4_lora", "lsi_m4_lorma", "lsi_kd_m1"]
MIN_MODELS = 2

SEED = 42
THRESH_MIN_POS = 20               # per-label thresholds only for statutes with at least this many positives
WEIGHT_GRID = [0.0, 0.5, 1.0, 1.5, 2.0]
WEIGHT_ROUNDS = 2
HELDOUT_REPEATS = 3               # random half/half splits for the honest check (each split is scored both ways)
THRESHOLD_GRID = np.arange(0.05, 0.951, 0.01)

START_TIME = time.time()
os.makedirs(OUT_DIR, exist_ok=True)
print("Config loaded.")

## 2. Data bundle (statute list and test document ids)

In [ ]:
def find_one_file(search_roots, file_name):
    for root in search_roots:
        for dir_path, dir_names, file_names in os.walk(root):
            if file_name in file_names:
                return os.path.join(dir_path, file_name)
    return None


if DATA_DIR is None:
    pkl_path = find_one_file([INPUT_ROOT, "/kaggle/working"], "tokenized_lsi.pkl")
    if pkl_path is None:
        raise FileNotFoundError("tokenized_lsi.pkl not found. Attach the LSI data-prep output.")
    DATA_DIR = os.path.dirname(pkl_path)
print("data dir:", DATA_DIR)

if os.path.exists(os.path.join(DATA_DIR, "lsi_utils.py")):
    shutil.copy(os.path.join(DATA_DIR, "lsi_utils.py"), "lsi_utils.py")
sys.path.insert(0, os.getcwd())
import lsi_utils

with open(os.path.join(DATA_DIR, "tokenized_lsi.pkl"), "rb") as f:
    DATA = pickle.load(f)

STATUTES = [int(s) for s in DATA["statute_ids"]]
N_STATUTES = len(STATUTES)
test_doc_ids = [str(x) for x in DATA["splits"]["test"]["doc_ids"]]
N_TEST = len(test_doc_ids)
del DATA
print("statutes:", N_STATUTES, "| test docs:", N_TEST)

## 3. Find the finished runs and line up their out-of-fold documents

In [ ]:
def folds_in_folder(folder):
    # Fold numbers for which all four files exist.
    names = set(os.listdir(folder))
    folds = []
    for k in range(10):
        needed = ["oof_probs_fold%d.npy" % k, "oof_labels_fold%d.npy" % k,
                  "oof_doc_ids_fold%d.json" % k, "test_probs_fold%d.npy" % k]
        has_all = True
        for file_name in needed:
            if file_name not in names:
                has_all = False
        if has_all:
            folds.append(k)
    return folds


def find_model_folder(name):
    # Folder named `name` with the most finished folds (newest wins a tie).
    best = None
    for dir_path, dir_names, file_names in os.walk(INPUT_ROOT):
        if os.path.basename(dir_path) != name:
            continue
        folds = folds_in_folder(dir_path)
        if len(folds) == 0:
            continue
        stamp = os.path.getmtime(os.path.join(dir_path, "oof_probs_fold%d.npy" % folds[0]))
        key = (len(folds), stamp)
        if best is None or key > best[0]:
            best = (key, dir_path, folds)
    if best is None:
        return None, []
    return best[1], best[2]


MODELS = {}
for name in MODEL_FOLDERS:
    folder, folds = find_model_folder(name)
    if folder is None:
        print(name.ljust(14), ": not found, skipped")
    else:
        MODELS[name] = {"dir": folder, "folds": folds}
        print(name.ljust(14), ": ", folder, "| folds", folds)

if len(MODELS) < MIN_MODELS:
    raise RuntimeError("Found %d finished runs, need at least %d. Attach more run outputs." % (len(MODELS), MIN_MODELS))

common_folds = None
for name in MODELS:
    fold_set = set(MODELS[name]["folds"])
    if common_folds is None:
        common_folds = fold_set
    else:
        common_folds = common_folds & fold_set
common_folds = sorted(common_folds)
if len(common_folds) == 0:
    raise RuntimeError("The runs share no fold. Folds per run: " + str({n: MODELS[n]["folds"] for n in MODELS}))
print("folds used (finished by every run):", common_folds)

NAMES = list(MODELS.keys())

# Load OOF rows of the shared folds, in the order of the first run, matching the others by document id.
ref_name = NAMES[0]
OOF_IDS = []
for k in common_folds:
    with open(os.path.join(MODELS[ref_name]["dir"], "oof_doc_ids_fold%d.json" % k)) as fh:
        OOF_IDS.extend([str(x) for x in json.load(fh)])
N_OOF = len(OOF_IDS)
if len(set(OOF_IDS)) != N_OOF:
    raise RuntimeError("Duplicate document ids in the OOF rows.")

OOF = {}
TEST = {}
Y_OOF = None
for name in NAMES:
    ids_m = []
    probs_m = []
    labels_m = []
    test_m = []
    for k in common_folds:
        with open(os.path.join(MODELS[name]["dir"], "oof_doc_ids_fold%d.json" % k)) as fh:
            ids_m.extend([str(x) for x in json.load(fh)])
        probs_m.append(np.load(os.path.join(MODELS[name]["dir"], "oof_probs_fold%d.npy" % k)))
        labels_m.append(np.load(os.path.join(MODELS[name]["dir"], "oof_labels_fold%d.npy" % k)))
        test_m.append(np.load(os.path.join(MODELS[name]["dir"], "test_probs_fold%d.npy" % k)))
    probs_m = np.concatenate(probs_m, axis=0).astype(np.float64)
    labels_m = np.concatenate(labels_m, axis=0).astype(np.int64)
    if set(ids_m) != set(OOF_IDS) or len(ids_m) != N_OOF:
        raise RuntimeError(name + ": its OOF documents differ from " + ref_name + ". The runs must use the same data bundle and folds.")
    row_of = {}
    for r in range(len(ids_m)):
        row_of[ids_m[r]] = r
    order = []
    for doc_id in OOF_IDS:
        order.append(row_of[doc_id])
    probs_m = probs_m[order]
    labels_m = labels_m[order]
    if probs_m.shape[1] != N_STATUTES:
        raise RuntimeError(name + ": probabilities have %d columns, expected %d" % (probs_m.shape[1], N_STATUTES))
    if Y_OOF is None:
        Y_OOF = labels_m
    elif not np.array_equal(Y_OOF, labels_m):
        raise RuntimeError(name + ": labels differ from " + ref_name + " for the same documents.")
    test_avg = np.mean(np.stack(test_m, axis=0), axis=0).astype(np.float64)
    if test_avg.shape != (N_TEST, N_STATUTES):
        raise RuntimeError(name + ": test probabilities have shape " + str(test_avg.shape) + ", expected " + str((N_TEST, N_STATUTES)))
    OOF[name] = probs_m
    TEST[name] = test_avg

Y_BOOL = Y_OOF.astype(bool)
print("OOF documents:", N_OOF, "| mean labels per doc:", round(float(Y_OOF.sum(axis=1).mean()), 2))

## 4. Fast metrics, thresholds, and ways to combine

In [ ]:
def f1_per_label(Y, pred):
    # Y, pred: bool [N, S]. F1 of every statute (0 when a statute has no true positive).
    tp = (pred & Y).sum(axis=0).astype(np.float64)
    fp = (pred & ~Y).sum(axis=0).astype(np.float64)
    fn = ((~pred) & Y).sum(axis=0).astype(np.float64)
    denom = 2 * tp + fp + fn
    f1 = np.zeros(len(tp))
    ok = tp > 0
    f1[ok] = 2 * tp[ok] / denom[ok]
    return f1


def macro_f1(Y, pred):
    return float(f1_per_label(Y, pred).mean())


def best_global(P, Y):
    best_t = 0.5
    best_f = -1.0
    for t in THRESHOLD_GRID:
        f = macro_f1(Y, P >= t)
        if f > best_f:
            best_f = f
            best_t = float(t)
    return best_t, best_f


def per_label_thresholds(P, Y, global_t):
    # One threshold per statute (best F1 of that statute). Rare statutes keep the global threshold.
    n_pos = Y.sum(axis=0)
    thr = np.full(P.shape[1], float(global_t))
    best_f = np.full(P.shape[1], -1.0)
    for t in THRESHOLD_GRID:
        f = f1_per_label(Y, P >= t)
        better = (f > best_f) & (n_pos >= THRESH_MIN_POS)
        thr[better] = float(t)
        best_f[better] = f[better]
    return thr


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def combine(P_list, weights, how):
    total = float(np.sum(weights))
    if how == "logit":
        z = np.zeros_like(P_list[0])
        for i in range(len(P_list)):
            z += weights[i] * logit(P_list[i])
        return 1.0 / (1.0 + np.exp(-z / total))
    out = np.zeros_like(P_list[0])
    for i in range(len(P_list)):
        out += weights[i] * P_list[i]
    return out / total


def tune_weights(P_list, Y):
    # Coordinate ascent on the macro-F1 at the best global threshold.
    weights = [1.0] * len(P_list)
    best_score = best_global(combine(P_list, weights, "prob"), Y)[1]
    for _ in range(WEIGHT_ROUNDS):
        for i in range(len(P_list)):
            for value in WEIGHT_GRID:
                trial = list(weights)
                trial[i] = value
                if sum(trial) <= 0:
                    continue
                score = best_global(combine(P_list, trial, "prob"), Y)[1]
                if score > best_score + 1e-6:
                    best_score = score
                    weights = trial
    return weights


# Candidates. "single" = one run alone, "mean_prob" = plain average, "mean_logit" = average in logit space,
# "weighted_prob" = weighted average with weights tuned on the tuning half only.
CANDIDATES = []
for name in NAMES:
    CANDIDATES.append(("single", name))
CANDIDATES.append(("mean_prob", None))
CANDIDATES.append(("mean_logit", None))
CANDIDATES.append(("weighted_prob", None))


def build(candidate, tune_rows, apply_rows):
    # Returns (ensemble probabilities on tune_rows, on apply_rows, weights used).
    kind, name = candidate
    if kind == "single":
        return OOF[name][tune_rows], OOF[name][apply_rows], None
    P_tune = [OOF[n][tune_rows] for n in NAMES]
    P_apply = [OOF[n][apply_rows] for n in NAMES]
    if kind == "weighted_prob":
        weights = tune_weights(P_tune, Y_BOOL[tune_rows])
        how = "prob"
    else:
        weights = [1.0] * len(NAMES)
        how = "prob" if kind == "mean_prob" else "logit"
    return combine(P_tune, weights, how), combine(P_apply, weights, how), weights


def heldout_scores(candidate):
    # Tune on one random half, score on the other half (and back). Returns {"global": mean, "per_label": mean}.
    rng = np.random.RandomState(SEED)
    got = {"global": [], "per_label": []}
    for rep in range(HELDOUT_REPEATS):
        perm = rng.permutation(N_OOF)
        half_a = np.sort(perm[: N_OOF // 2])
        half_b = np.sort(perm[N_OOF // 2:])
        for tune_rows, test_rows in [(half_a, half_b), (half_b, half_a)]:
            P_tune, P_test, _ = build(candidate, tune_rows, test_rows)
            t_glob, _ = best_global(P_tune, Y_BOOL[tune_rows])
            got["global"].append(macro_f1(Y_BOOL[test_rows], P_test >= t_glob))
            thr = per_label_thresholds(P_tune, Y_BOOL[tune_rows], t_glob)
            got["per_label"].append(macro_f1(Y_BOOL[test_rows], P_test >= thr))
    return {"global": float(np.mean(got["global"])), "per_label": float(np.mean(got["per_label"]))}


print("Helpers ready.")

## 5. Score every candidate on documents it was not tuned on

In [ ]:
RESULTS = []
for candidate in CANDIDATES:
    t0 = time.time()
    scores = heldout_scores(candidate)
    label = candidate[0] if candidate[1] is None else candidate[0] + ":" + candidate[1]
    RESULTS.append({"candidate": candidate, "label": label, "global": scores["global"], "per_label": scores["per_label"]})
    print(label.ljust(24), "| held-out macro-F1  global threshold:", round(scores["global"], 4),
          "| per-label thresholds:", round(scores["per_label"], 4), "|", round(time.time() - t0, 1), "s")

best = None
for r in RESULTS:
    for mode in ["global", "per_label"]:
        if best is None or r[mode] > best["score"]:
            best = {"candidate": r["candidate"], "label": r["label"], "mode": mode, "score": r[mode]}
print("\nbest on held-out documents:", best["label"], "+", best["mode"], "thresholds | macro-F1", round(best["score"], 4))

best_single = max(RESULTS, key=lambda r: max(r["global"], r["per_label"]) if r["candidate"][0] == "single" else -1.0)
print("best single run:", best_single["label"], "| macro-F1", round(max(best_single["global"], best_single["per_label"]), 4))

## 6. Fit the winner on all OOF documents and write the submission

In [ ]:
all_rows = np.arange(N_OOF)
kind, name = best["candidate"]

if kind == "single":
    P_oof = OOF[name]
    P_test = TEST[name]
    weights = None
else:
    if kind == "weighted_prob":
        weights = tune_weights([OOF[n] for n in NAMES], Y_BOOL)
        how = "prob"
    else:
        weights = [1.0] * len(NAMES)
        how = "prob" if kind == "mean_prob" else "logit"
    P_oof = combine([OOF[n] for n in NAMES], weights, how)
    P_test = combine([TEST[n] for n in NAMES], weights, how)
    print("weights:", {NAMES[i]: weights[i] for i in range(len(NAMES))})

global_t, global_macro = best_global(P_oof, Y_BOOL)
if best["mode"] == "per_label":
    thr = per_label_thresholds(P_oof, Y_BOOL, global_t)
else:
    thr = np.full(N_STATUTES, global_t)
in_sample = macro_f1(Y_BOOL, P_oof >= thr)
print("global threshold:", round(global_t, 2), "| in-sample OOF macro-F1:", round(in_sample, 4),
      "(optimistic) | held-out estimate:", round(best["score"], 4), "(honest)")

pred_lists = []
for i in range(N_TEST):
    row = P_test[i]
    chosen = []
    for j in range(N_STATUTES):
        if row[j] >= thr[j]:
            chosen.append(STATUTES[j])
    if len(chosen) == 0:
        chosen = [STATUTES[int(np.argmax(row))]]       # every training document has at least one statute
    pred_lists.append(chosen)

sub_path = os.path.join(OUT_DIR, "submission_lsi_ensemble.json")
lsi_utils.write_submission(test_doc_ids, pred_lists, STATUTES, sub_path)

# Codabench upload: prediction.json inside submission.zip (the same packaging as the Phase 3 notebooks).
pred_path = os.path.join(OUT_DIR, "prediction.json")
shutil.copy(sub_path, pred_path)
with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(pred_path, "prediction.json")

with open(os.path.join(OUT_DIR, "per_label_thresholds.json"), "w") as f:
    json.dump({str(STATUTES[j]): float(thr[j]) for j in range(N_STATUTES)}, f, indent=1)
np.save(os.path.join(OUT_DIR, "ensemble_test_probs.npy"), P_test.astype(np.float32))

table = []
for r in RESULTS:
    table.append({"candidate": r["label"], "heldout_global": r["global"], "heldout_per_label": r["per_label"]})
report = {
    "models": {n: {"dir": MODELS[n]["dir"], "folds": MODELS[n]["folds"]} for n in NAMES},
    "folds_used": common_folds,
    "oof_documents": N_OOF,
    "heldout_table": table,
    "winner": {"candidate": best["label"], "thresholds": best["mode"], "heldout_macro_f1": best["score"]},
    "weights": None if weights is None else {NAMES[i]: weights[i] for i in range(len(NAMES))},
    "global_threshold": global_t,
    "in_sample_oof_macro_f1": in_sample,
}
with open(os.path.join(OUT_DIR, "ensemble_report.json"), "w") as f:
    json.dump(report, f, indent=1)
print("report saved | elapsed (s):", round(time.time() - START_TIME, 1))

## 7. Check the files

In [ ]:
for file_name in sorted(os.listdir(OUT_DIR)):
    full = os.path.join(OUT_DIR, file_name)
    if os.path.isfile(full):
        print(file_name.ljust(32), round(os.path.getsize(full) / 1e6, 2), "MB")

sub = json.load(open(os.path.join(OUT_DIR, "submission_lsi_ensemble.json")))
assert len(sub) == N_TEST
allowed = set(STATUTES)
for i in range(len(sub)):
    row = sub[i]
    assert str(row["id"]) == test_doc_ids[i], "submission order differs from the test document order"
    assert len(row["pred"]) > 0 and len(row["pred"]) == len(set(row["pred"]))
    for p in row["pred"]:
        assert int(p) in allowed
with zipfile.ZipFile(os.path.join(OUT_DIR, "submission.zip")) as zf:
    assert zf.namelist() == ["prediction.json"]
print("submission rows:", len(sub), "| ids, order, statutes and zip checked")
print("Done.")

## Notes
- Held-out check: weights and thresholds are tuned on one random half of the OOF documents and scored on the other half (3 random splits, both directions). The winner is the candidate with the best held-out macro-F1, so a fancier combination only wins if it really generalises.
- If only fold 0 has finished, the OOF set is that fold's validation documents. Test probabilities are the average over the folds that finished.
- Upload `submission.zip`. If your earlier accepted zips used another file name inside, rename `prediction.json` to match.